In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported!")

All libraries imported!


In [22]:
df = pd.read_csv("../Data/credit_data.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Shape: (1000, 10)

Columns: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']

First 5 rows:
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own             NaN           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little              NaN           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  Risk  
0         6             radio/TV  good  
1        48             radio/TV   bad  
2        12            education  good  
3        42  furniture/equipment   bad  
4        24                  car   bad  


In [23]:
print("Missing Values BEFORE:")
print(df.isnull().sum())

# Fill missing with mode (most frequent value)
df['Saving accounts'] = df['Saving accounts'].fillna(df['Saving accounts'].mode()[0])
df['Checking account'] = df['Checking account'].fillna(df['Checking account'].mode()[0])

print("\nMissing Values AFTER:")
print(df.isnull().sum())
print("\nAll missing values handled!")

Missing Values BEFORE:
Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     183
Checking account    394
Credit amount         0
Duration              0
Purpose               0
Risk                  0
dtype: int64

Missing Values AFTER:
Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64

All missing values handled!


In [7]:
# Convert Risk to binary: good=0, bad=1
df['Risk'] = df['Risk'].map({'good': 0, 'bad': 1})

print("Target Variable Encoded:")
print(df['Risk'].value_counts())
print(f"\ngood = 0, bad = 1")

Target Variable Encoded:
Risk
0    522
1    478
Name: count, dtype: int64

good = 0, bad = 1


In [8]:
le = LabelEncoder()

categorical_cols = ['Sex', 'Housing', 'Saving accounts', 
                    'Checking account', 'Purpose']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])
    print(f"{col} encoded!")

print("\nData after encoding:")
df.head()

Sex encoded!
Housing encoded!
Saving accounts encoded!
Checking account encoded!
Purpose encoded!

Data after encoding:


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,1,2,1,0,0,1169,6,5,0
1,22,0,2,1,0,1,5951,48,5,1
2,49,1,1,1,0,0,2096,12,3,0
3,45,1,2,0,0,0,7882,42,4,1
4,53,1,2,0,0,0,4870,24,1,1


In [9]:
X = df.drop('Risk', axis=1)
y = df['Risk']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:", X.columns.tolist())
print("\nClass Distribution:")
print(y.value_counts())

Features shape: (1000, 9)
Target shape: (1000,)

Features: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']

Class Distribution:
Risk
0    522
1    478
Name: count, dtype: int64


In [10]:
scaler = StandardScaler()

numerical_cols = ['Age', 'Credit amount', 'Duration']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Features after scaling:")
X.head()

Features after scaling:


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,2.766456,1,2,1,0,0,-0.745131,-1.236478,5
1,-1.191404,0,2,1,0,1,0.949817,2.248194,5
2,1.183312,1,1,1,0,0,-0.416562,-0.738668,3
3,0.831502,1,2,0,0,0,1.634247,1.750384,4
4,1.535122,1,2,0,0,0,0.566664,0.256953,1


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("=== Data Split ===")
print(f"Training set   : {X_train.shape}")
print(f"Testing set    : {X_test.shape}")
print(f"\nTraining Target Distribution:")
print(y_train.value_counts())
print(f"\nTesting Target Distribution:")
print(y_test.value_counts())

=== Data Split ===
Training set   : (800, 9)
Testing set    : (200, 9)

Training Target Distribution:
Risk
0    418
1    382
Name: count, dtype: int64

Testing Target Distribution:
Risk
0    104
1     96
Name: count, dtype: int64


In [13]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("=== After SMOTE ===")
print(f"Training set before SMOTE: {X_train.shape}")
print(f"Training set after SMOTE : {X_train_resampled.shape}")
print(f"\nClass Distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("\nDataset is now balanced!")

=== After SMOTE ===
Training set before SMOTE: (800, 9)
Training set after SMOTE : (836, 9)

Class Distribution after SMOTE:
Risk
1    418
0    418
Name: count, dtype: int64

Dataset is now balanced!


In [17]:
import os
import pandas as pd

# Define processed data folder
processed_path = r"C:\Users\E106006\OneDrive - Road Accident Fund\Desktop\github Portfolio\Credit Risk\Data\processed"

# Create folder if it doesn't exist
os.makedirs(processed_path, exist_ok=True)

# Convert arrays back to DataFrames
X_train_resampled_df = pd.DataFrame(X_train_resampled, columns=X.columns)
X_train_resampled_df["Risk"] = y_train_resampled

X_test_df = pd.DataFrame(X_test, columns=X.columns)
X_test_df["Risk"] = y_test.reset_index(drop=True)

# Save datasets
train_file = os.path.join(processed_path, "train_data.csv")
test_file = os.path.join(processed_path, "test_data.csv")

X_train_resampled_df.to_csv(train_file, index=False)
X_test_df.to_csv(test_file, index=False)

print("=== Saved Successfully! ===")
print(f"Train data shape: {X_train_resampled_df.shape}")
print(f"Test data shape : {X_test_df.shape}")
print(f"\nTrain file: {train_file}")
print(f"Test file : {test_file}")

=== Saved Successfully! ===
Train data shape: (836, 10)
Test data shape : (200, 10)

Train file: C:\Users\E106006\OneDrive - Road Accident Fund\Desktop\github Portfolio\Credit Risk\Data\processed\train_data.csv
Test file : C:\Users\E106006\OneDrive - Road Accident Fund\Desktop\github Portfolio\Credit Risk\Data\processed\test_data.csv


In [18]:
print("=" * 50)
print("     PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Original dataset shape  : (1000, 10)")
print(f"After encoding          : All categories numeric")
print(f"Missing values handled  : Filled with mode")
print(f"Train size              : {X_train_resampled.shape[0]} rows")
print(f"Test size               : {X_test.shape[0]} rows")
print(f"After SMOTE             : Classes balanced 50/50")
print(f"\nNext Step: Model Training!")

     PREPROCESSING SUMMARY
Original dataset shape  : (1000, 10)
After encoding          : All categories numeric
Missing values handled  : Filled with mode
Train size              : 836 rows
Test size               : 200 rows
After SMOTE             : Classes balanced 50/50

Next Step: Model Training!


In [20]:
import joblib

# Save scaler
joblib.dump(
    scaler,
    '../models/scaler.pkl'
)

print("Scaler saved successfully!")

Scaler saved successfully!
